# Étape 4 — Collecte des hébergements (Booking.com)

## Objectif

Récupérer environ 15 hébergements par destination du Top 5, avec nom, URL, note,
nombre d'avis, description, coordonnées GPS et tarifs.

## Entrées / sorties

| | |
|---|---|
| **Entrée** | `data/processed/top5_destinations.csv` |
| **Sorties** | `data/raw/hotels_json/*.json` (réponses brutes), `data/raw/hotels/*.csv`, `data/raw/hotels_top5_all.csv` |
| **Service** | [BrightData](https://brightdata.com/) — collecteur Booking.com managé |

## Pourquoi un service tiers plutôt qu'un scraper maison

Booking.com applique des protections anti-bot (paramètre `chal_t` horodaté dans les URLs,
détection de navigateur, blocage d'IP). Un scraper Selenium tiendrait quelques dizaines de
requêtes avant d'être bloqué. BrightData gère la rotation d'IP et le rendu, et renvoie du
JSON structuré. C'est bien du **scraping**, comme demandé par l'énoncé — mais délégué.

## Architecture en deux temps

Le collecteur est **asynchrone** : on ne peut pas demander les données et les recevoir
dans le même appel.

1. **`src/trigger_scraping.py`** — un `POST /trigger` par ville, lancés en parallèle.
   Chaque appel renvoie un `snapshot_id`, enregistré dans un registre JSON.
2. **`src/fetch_results.py`** — un `GET /snapshot/{id}` par ville, en *polling* jusqu'à ce
   que les données soient prêtes, puis parsing en DataFrame.

Séparer les deux permet de relancer la récupération sans re-déclencher (et re-payer) le
scraping.

## Paramètres de la recherche

| Paramètre | Valeur | Justification |
|---|---|---|
| Arrivée | **J+30** | à un mois, les tarifs sont disponibles et stables ; à J+3 beaucoup d'établissements sont complets |
| Durée | 2 nuits | un week-end, l'unité de séjour la plus courante |
| Occupants | 2 adultes, 1 chambre | référence de comparaison entre établissements |

⚠️ **Les deux fenêtres temporelles du projet ne coïncident donc pas** : la météo porte sur
les 6 prochains jours, les tarifs sur un séjour dans un mois. C'est un choix assumé,
signalé dans le rapport final.

## ⚠️ Coût

Chaque exécution déclenche une collecte **facturée** chez BrightData. Les réponses JSON
brutes sont versionnées dans `data/raw/hotels_json/` précisément pour pouvoir rejouer le
parsing sans relancer de collecte : `python src/reparse_hotels.py`.

In [4]:
# ════════════════════════════════════════════════════════════════════
# CELLULE 1 : Imports et configuration
# ════════════════════════════════════════════════════════════════════

import sys
sys.path.append('../src')

import pandas as pd
import nest_asyncio
from datetime import datetime

# Permettre asyncio dans Jupyter
nest_asyncio.apply()

print("✅ Configuration prête")

✅ Configuration prête


In [3]:
# ════════════════════════════════════════════════════════════════════
# CELLULE 2 : STEP 1 - Déclencher les scrapings (POST)
# ════════════════════════════════════════════════════════════════════

import trigger_scraping as step1

# Charger le Top 5
top5 = pd.read_csv('../data/processed/top5_destinations.csv')
cities_list = top5['city'].tolist()

print("🏆 Top 5 des destinations :")
for i, city in enumerate(cities_list, 1):
    print(f"   {i}. {city}")

# Configuration
MAX_HOTELS = 15

print(f"\n🚀 Lancement STEP 1...")
print(f"   • {MAX_HOTELS} hôtels/ville")
print(f"   • {len(cities_list)} villes\n")

# LANCER STEP 1
registry = await step1.trigger_all_cities(cities_list, MAX_HOTELS)

print("✅ STEP 1 terminé - Snapshots enregistrés !")

🏆 Top 5 des destinations :
   1. Aix en Provence
   2. Marseille
   3. Bormes les Mimosas
   4. Cassis
   5. Avignon

🚀 Lancement STEP 1...
   • 15 hôtels/ville
   • 5 villes


📤 STEP 1 : DÉCLENCHEMENT DES SCRAPINGS (POST)
🏙️  Villes : 5
🏨 Hôtels par ville : 15
⏱️  Démarrage : 19:32:21

🔍 Chargement depuis : c:\Users\Emeline\Documents\_DEV\6_Projet_Kayak\kayak_project\notebooks\..\config\.env
✅ API Key chargée : 50cd2e58bb4e001874f0...
✅ Bormes les Mimosas   → Snapshot: sd_mhuwrnki2j0jgytc6x
✅ Marseille            → Snapshot: sd_mhuwrnla1i698hvu25
✅ Cassis               → Snapshot: sd_mhuwrnm31jdc1323zf
✅ Avignon              → Snapshot: sd_mhuwrnmq2bhcbfb0gr
✅ Aix en Provence      → Snapshot: sd_mhuwrnms1vhvy0jx7g
💾 Registre sauvegardé : data/raw/snapshots/snapshots_registry.json

✅ STEP 1 TERMINÉ
📊 Snapshots créés : 5/5
📁 Registre : data/raw/snapshots/snapshots_registry.json

✅ STEP 1 terminé - Snapshots enregistrés !


In [5]:
# ════════════════════════════════════════════════════════════════════
# CELLULE 3 : STEP 2 - Récupération des résultats (GET)
# ════════════════════════════════════════════════════════════════════

import fetch_results as step2

# Recharger le module (au cas où vous l'avez modifié)
import importlib
importlib.reload(step2)

print("🚀 Lancement STEP 2...")
print("⏳ Cela peut prendre 10-15 minutes...\n")

# LANCER STEP 2
results = await step2.fetch_all_results()

print(f"\n✅ STEP 2 terminé - {len(results)} villes récupérées !")

🚀 Lancement STEP 2...
⏳ Cela peut prendre 10-15 minutes...


📥 STEP 2 : RÉCUPÉRATION DES RÉSULTATS (GET)
⏱️  Démarrage : 21:02:23

📊 5 snapshot(s) à récupérer :
   • Aix en Provence           → sd_mhuwrnms1vhvy0jx7g
   • Marseille                 → sd_mhuwrnla1i698hvu25
   • Bormes les Mimosas        → sd_mhuwrnki2j0jgytc6x
   • Cassis                    → sd_mhuwrnm31jdc1323zf
   • Avignon                   → sd_mhuwrnmq2bhcbfb0gr

✅ API Key chargée : 50cd2e58bb4e001874f0...
⏳ Récupération en cours...

✅ Bormes les Mimosas   → 15 hôtels (0s)
   💾 JSON brut : data/raw/hotels_json/bormes_les_mimosas_raw.json
✅ Cassis               → 15 hôtels (0s)
   💾 JSON brut : data/raw/hotels_json/cassis_raw.json
✅ Aix en Provence      → 15 hôtels (0s)
   💾 JSON brut : data/raw/hotels_json/aix_en_provence_raw.json
✅ Avignon              → 15 hôtels (0s)
   💾 JSON brut : data/raw/hotels_json/avignon_raw.json
✅ Marseille            → 15 hôtels (0s)
   💾 JSON brut : data/raw/hotels_json/marseille_raw.j

## Contrôle du résultat

Vérification du volume collecté et de la complétude avant de passer au nettoyage.

In [14]:
# ════════════════════════════════════════════════════════════════════
# CELLULE 4 : Statistiques
# ════════════════════════════════════════════════════════════════════

# Charger les données
all_hotels = pd.read_csv('../data/raw/hotels_top5_all.csv')

print(f"\n{'='*80}")
print("📊 STATISTIQUES FINALES")
print(f"{'='*80}\n")

print(f"Total hôtels : {len(all_hotels)}")
print(f"Score moyen : {all_hotels['score'].mean():.2f}/10")

if all_hotels['price'].notna().any():
    print(f"Prix moyen : {all_hotels['price'].mean():.0f} EUR")

print(f"\nPar ville :")
stats = all_hotels.groupby('city').size()
for city, count in stats.items():
    print(f"   • {city:20s} : {count} hôtels")

print(f"\n🔍 Aperçu :")
display(all_hotels[['city', 'hotel_name', 'score', 'price']].head(10))


📊 STATISTIQUES FINALES

Total hôtels : 75
Score moyen : 8.33/10
Prix moyen : 329 EUR

Par ville :
   • Aix en Provence      : 15 hôtels
   • Avignon              : 15 hôtels
   • Bormes les Mimosas   : 15 hôtels
   • Cassis               : 15 hôtels
   • Marseille            : 15 hôtels

🔍 Aperçu :


,city,hotel_name,score,price
0,Aix en Provence,Aparthotel Adagio Aix-en-Provence Centre,8.5,221.0
1,Aix en Provence,Best Western Hotel le Galice Aix-en-Provence,8.0,233.0
2,Aix en Provence,B&B HOTEL Aix-en-Provence Pont de l'Arc,7.5,128.0
3,Aix en Provence,Villa Saint-Ange,9.3,752.0
4,Aix en Provence,Le Concorde,8.0,260.0
5,Aix en Provence,Les Suites du Cours & Spa,9.2,398.0
6,Aix en Provence,Hotel Cardinal,8.6,313.0
7,Aix en Provence,Campanile Prime - Aix-en-Provence Sud - Pont d...,7.8,155.0
8,Aix en Provence,Domaine de Saint Clair,9.6,561.0
9,Aix en Provence,Adonis Arc Hotel Aix,7.6,160.0


## Conclusion

**75 hébergements collectés**, 15 par destination, avec 100 % de coordonnées GPS.

Trois défauts des données brutes, corrigés à l'étape suivante et au parsing :

1. BrightData ignore le paramètre `currency: EUR` et renvoie les montants **en USD**, pour
   la **durée totale du séjour** (2 nuits) et non par nuit ;
2. un établissement sans avis arrive avec `review_score: 0`, qui n'est pas une note de
   0/10 mais une note **absente** ;
3. un doublon existe dans les résultats de Marseille — les deux URLs diffèrent par le
   paramètre anti-bot `chal_t`, seul `listing_id` permet de le détecter.

➡️ Étape suivante : `05_hotels_cleaning.ipynb`.

> ⚠️ **Ce notebook n'est pas rejouable sans coût.** Chaque exécution déclenche une
> collecte facturée chez BrightData, et les numéros d'exécution affichés proviennent de la
> session de collecte d'origine. Pour rejouer uniquement le **parsing** à partir des
> réponses JSON versionnées, sans nouvelle collecte :
> ```bash
> python src/reparse_hotels.py
> ```